In [5]:
import os


In [30]:
#Part 1: Setting Up the Database

#Create database.py and add:

import sqlite3

def setup_database():
    conn = sqlite3.connect('company.db')
    c = conn.cursor()

    # Create sample tables
    c.execute('''
        CREATE TABLE IF NOT EXISTS employees (
            id INTEGER PRIMARY KEY,
            name TEXT,
            department TEXT,
            salary REAL
        )
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS departments (
            id INTEGER PRIMARY KEY,
            name TEXT,
            budget REAL
        )
    ''')

    # Insert sample data
    c.execute("INSERT OR IGNORE INTO employees VALUES (1, 'John Doe', 'Engineering', 75000)")
    c.execute("INSERT OR IGNORE INTO employees VALUES (2, 'Jane Smith', 'Marketing', 65000)")
    c.execute("INSERT OR IGNORE INTO departments VALUES (1, 'Engineering', 1000000)")
    c.execute("INSERT OR IGNORE INTO departments VALUES (2, 'Marketing', 500000)")

    conn.commit()
    conn.close()

setup_database()
conn = sqlite3.connect('company.db')
c = conn.cursor()

print("Employees table:")
c.execute("SELECT * FROM employees")
print(c.fetchall())

print("\nDepartments table:")
c.execute("SELECT * FROM departments")
print(c.fetchall())

Employees table:
[(1, 'John Doe', 'Engineering', 75000.0), (2, 'Jane Smith', 'Marketing', 65000.0)]

Departments table:
[(1, 'Engineering', 1000000.0), (2, 'Marketing', 500000.0)]


In [8]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (1,634 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current u

In [9]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [10]:
!ollama -v
!ollama pull llama3.2
!ollama list

ollama version is 0.20.2

NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [64]:
# Adding OpenAI Function Definition
import openai
from openai import OpenAI

class SQLHelper:
    """
    A helper class for generating, validating, executing, and formatting SQL queries
    using a local Ollama server running Llama3.2.
    """
    def __init__(self):
        """
        Initializes the SQLHelper with an OpenAI client configured to connect
        to a local Ollama server.
        """
        # Using OpenAI client to interact with the local Ollama server running Llama3.2
        self.client = OpenAI(
            base_url='http://localhost:11434/v1/',
            api_key='ollama', # Required but ignored by Ollama
        )

    def get_schema(self):
        """
        Returns the database schema as a string, describing the available tables and columns.

        Returns:
            str: A string representation of the database schema.
        """
        return """
        Table: employees
        Columns:
        - id (INTEGER PRIMARY KEY)
        - name (TEXT)
        - department (TEXT)
        - salary (REAL)

        Table: departments
        Columns:
        - id (INTEGER PRIMARY KEY)
        - name (TEXT)
        - budget (REAL)
        """

    def generate_sql(self, question):
        """
        Generates a SQL query based on a natural language question using the Llama3.2 model.

        Args:
            question (str): The natural language question to convert into SQL.

        Returns:
            str: The generated SQL query, or None if an error occurs.
        """
        try:
            response = self.client.chat.completions.create(
                model="llama3.2",
                messages=[
                    {"role": "system", "content": f"""You are a SQL expert. Use this schema:\n{self.get_schema()}
                    When joining tables, ensure you use logically related columns with compatible data types. For example, to relate employees to departments, join 'employees.department' with 'departments.name'.
                    Return only one best performing SQL query without any explanation or markdown formatting."""},
                    {"role": "user", "content": f"Generate SQL for: {question}"}
                ],
            )
            sql = response.choices[0].message.content.strip()

            # Remove any markdown code block syntax generated by the LLM
            sql = sql.replace('```sql', '').replace('```SQL', '').replace('```', '')

            # If there's a semicolon followed by an explanation, remove the explanation
            if ';' in sql:
                sql = sql.split(';')[0].strip()

            # Remove any explanatory text before or after the SQL (redundant with above, but good as fallback)
            sql_lines = [line.strip() for line in sql.split('\n') if line.strip()]
            sql = ' '.join(sql_lines)

            return sql
        except Exception as e:
            print(f"Error generating SQL: {e}")
            return None

    def validate_sql(self, sql):
        """
        Validates the generated SQL query to ensure it only contains SELECT statements.
        This prevents potentially dangerous operations like DROP, DELETE, UPDATE, or INSERT.

        Args:
            sql (str): The SQL query to validate.

        Returns:
            str: The validated SQL query.

        Raises:
            ValueError: If the SQL query contains disallowed keywords.
        """
        if not sql:
            raise ValueError("SQL query cannot be empty")

        sql_lower = sql.lower()
        # Basic safety checks: only SELECT queries are allowed
        if any(word in sql_lower for word in ['drop', 'delete', 'update', 'insert', 'alter', 'truncate']):
            raise ValueError("Only SELECT queries are allowed")
        return sql

    def execute_query(self, sql):
        """
        Executes a validated SQL query against the SQLite database.

        Args:
            sql (str): The SQL query to execute.

        Returns:
            list or str: A list of tuples representing the query results, or an error message string.
        """
        try:
            validated_sql = self.validate_sql(sql)
            conn = sqlite3.connect('company.db')
            cursor = conn.cursor()
            cursor.execute(validated_sql)
            results = cursor.fetchall()
            return results
        except ValueError as ve:
            return f"Validation Error: {str(ve)}"
        except sqlite3.Error as sqle:
            return f"Database Error: {str(sqle)}"
        except Exception as e:
            return f"An unexpected error occurred during query execution: {str(e)}"
        finally:
            if 'conn' in locals() and conn:
                conn.close()

    def format_results(self, results):
        """
        Formats the query results into a human-readable string representation.
        Handles single-column results, multi-column tabular data, and empty results.

        Args:
            results (list): A list of tuples containing the query results.

        Returns:
            str: A formatted string of the query results.
        """
        if not isinstance(results, list):
            return str(results) # Return as string if it's an error message or non-list
        if not results:
            return "No results found"

        # If it's a single column result
        if len(results[0]) == 1:
            return "\n".join([str(row[0]) for row in results])

        # For multiple columns, format as a table-like string
        formatted_rows = []
        for row in results:
            row_items = []
            for item in row:
                if isinstance(item, float):
                    # Format currency-like floats, otherwise general floats
                    row_items.append(f"${item:,.2f}" if any(col_name in str(row).lower() for col_name in ['salary', 'budget']) else f"{item:.2f}")
                else:
                    row_items.append(str(item))
            formatted_rows.append("\t".join(row_items))
        return "\n".join(formatted_rows)


In [65]:
#Test Part 2:

# Test SQL generation
test_questions = [
    "List all employees",
    "Show departments with budgets over 750000",
    "Which department has highest budget"
]
sql_helper = SQLHelper()
for question in test_questions:
    print(f"\nQuestion: {question}")
    print(f"Generated SQL: {sql_helper.generate_sql(question)}")


Question: List all employees
Generated SQL: SELECT id, name, department, salary FROM employees

Question: Show departments with budgets over 750000
Generated SQL: SELECT name FROM departments WHERE budget > 750000

Question: Which department has highest budget
Generated SQL: SELECT name, budget FROM departments ORDER BY budget DESC LIMIT 1


In [62]:
#Test Part 3:

# Test query execution
if __name__ == "__main__":
    test_sql = [
        "SELECT * FROM employees WHERE salary > 70000",
        "SELECT d.name, COUNT(e.id) FROM departments d LEFT JOIN employees e ON d.name = e.department GROUP BY d.name"
    ]

    for sql in test_sql:
        print(f"\nExecuting SQL: {sql}")
        print(f"Results: {sql_helper.execute_query(sql)}")


Executing SQL: SELECT * FROM employees WHERE salary > 70000
Results: [(1, 'John Doe', 'Engineering', 75000.0)]

Executing SQL: SELECT d.name, COUNT(e.id) FROM departments d LEFT JOIN employees e ON d.name = e.department GROUP BY d.name
Results: [('Engineering', 1), ('Marketing', 1)]


In [66]:
#Part 4: Creating the Complete Agent
import time
#Add the final agent code:
def query_agent(question):
  try:
      start_time = time.time()
      # Generate SQL
      sql = sql_helper.generate_sql(question)
      print(f"Generated SQL: {sql}\n")

      exec_start = time.time()
      # Execute and format results
      results = sql_helper.execute_query(sql)
      exec_time = time.time() - exec_start
      total_time = time.time() - start_time
      results = sql_helper.format_results(results)
      return results, exec_time, total_time
  except Exception as e:
      return f"Error: {str(e)}", None, None

# Test the complete agent
test_questions = [
    "What is the average salary in each department?",
    "Which department has the highest budget?",
    "List all employees earning more than 70000",
    "DROP TABLE employees"  # This should be caught by validation
]

for question in test_questions:
    print(f"\nQuestion: {question}")
    results, exec_time, total_time = query_agent(question)
    print(f"Answer: {results}, Query Execution Time: {exec_time}, Total Time: {total_time}")


Question: What is the average salary in each department?
Generated SQL: SELECT d.name AS department, AVG(e.salary) AS average_salary FROM employees e JOIN departments d ON e.department = d.id GROUP BY d.name ORDER BY average_salary DESC

Answer: No results found, Query Execution Time: 0.0006825923919677734, Total Time: 23.22286081314087

Question: Which department has the highest budget?
Generated SQL: SELECT d.name, d.budget FROM departments d ORDER BY d.budget DESC LIMIT 1

Answer: Engineering	1000000.00, Query Execution Time: 0.0006186962127685547, Total Time: 28.68609619140625

Question: List all employees earning more than 70000
Generated SQL: SELECT name, salary FROM employees WHERE salary > 70000

Answer: John Doe	75000.00, Query Execution Time: 0.0005645751953125, Total Time: 10.568708419799805

Question: DROP TABLE employees
Generated SQL: DROP TABLE employees

Answer: Validation Error: Only SELECT queries are allowed, Query Execution Time: 2.2649765014648438e-05, Total Time: